In [1]:
import sys; sys.path.append("/home/jovyan/work")
from pyspark.sql import functions as F
from commun.spark_utils import creer_session, RAW, LAKE

from commun.spark_utils import creer_bucket, format_tables
creer_bucket("datalake")

Bucket « datalake » créé.


In [2]:
spark = creer_session("S01b")
FMT = format_tables(spark)   # 'delta' (ou 'parquet' si Delta indisponible)
FMT

Préparation de Spark 4.2.0 (Hadoop 3.5.0) + Delta Lake + data lake S3 … le premier démarrage peut prendre quelques minutes.
Session prête. Spark UI : http://localhost:4040


'delta'

In [20]:

# Copie du CSV vers RustFS

RAW = "/home/jovyan/work/data/raw"
fichier = f"{RAW}/comptage_multimodal_comptages.csv"

# Lecture du CSV
comptages_raw = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .csv(fichier)
    .withColumn("_source", F.lit("comptages_multimodal_comptages.csv"))
    .withColumn("_ingere_le", F.current_timestamp())
)

# Renommer les colonnes pour qu'elles soient acceptées par Delta
comptages_raw = (
    comptages_raw
    .withColumnRenamed("Identifiant trajectoire", "identifiant_trajectoire")
    .withColumnRenamed("Identifiant site", "identifiant_site")
    .withColumnRenamed("Nom du site", "nom_du_site")
    .withColumnRenamed("Date et heure de comptage", "date_heure_comptage")
    .withColumnRenamed("Mode déplacement", "mode_deplacement")
    .withColumnRenamed("Nombre de véhicules", "nombre_de_vehicules")
    .withColumnRenamed("Type voie", "type_voie")
    .withColumnRenamed("Sens de circulation", "sens_circulation")
    .withColumnRenamed("Coordonnées Géo", "coordonnees_geo")
)

# Vérifier les colonnes et les données
print(comptages_raw.columns)
comptages_raw.show(3, truncate=False)

# Écriture dans RustFS au format Delta
chemin_bronze = f"{LAKE}/bronze/comptage_multimodal_paris"

(
    comptages_raw.write
    .format(FMT)
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(chemin_bronze)
)

bronze = spark.read.format(FMT).load(chemin_bronze)
print("Ingestion Bronze terminée.")



['identifiant_trajectoire', 'identifiant_site', 'nom_du_site', 'date_heure_comptage', 'mode_deplacement', 'nombre_de_vehicules', 'type_voie', 'sens_circulation', 'Trajectoire', 'coordonnees_geo', '_source', '_ingere_le']
+-----------------------+----------------+---------------------------+-------------------------+--------------------+-------------------+----------------------------+----------------+-----------+---------------------+----------------------------------+--------------------------+
|identifiant_trajectoire|identifiant_site|nom_du_site                |date_heure_comptage      |mode_deplacement    |nombre_de_vehicules|type_voie                   |sens_circulation|Trajectoire|coordonnees_geo      |_source                           |_ingere_le                |
+-----------------------+----------------+---------------------------+-------------------------+--------------------+-------------------+----------------------------+----------------+-----------+---------------------+--